In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Find the dataset folder from the project root or this notebook folder
dataset_root = Path("LP IV Updated Datasets")
if not dataset_root.is_dir():
    dataset_root = Path("../LP IV Updated Datasets")
print("Dataset folder:", dataset_root.resolve())


In [ ]:
# Load data
train = pd.read_csv(dataset_root / "FNN and CNN/Fashion MNIST/fashion-mnist_train.csv")
print("Training CSV shape:", train.shape)
print("Training preview (3 rows, 8 columns):\n", train.iloc[:3, :8])
print("Training label counts:\n", train["label"].value_counts().sort_index())
test = pd.read_csv(dataset_root / "FNN and CNN/Fashion MNIST/fashion-mnist_test.csv")
print("Test CSV shape:", test.shape)
print("Test preview (3 rows, 8 columns):\n", test.iloc[:3, :8])
print("Test label counts:\n", test["label"].value_counts().sort_index())


In [ ]:
# Prepare features and labels
x_train = train.iloc[0:, 1:] / 255
y_train = train.iloc[0:, 0]
x_test = test.iloc[0:, 1:] / 255
y_test = test.iloc[0:, 0]

print("Training features:", x_train.shape, "Training labels:", y_train.shape)
print("Test features:", x_test.shape, "Test labels:", y_test.shape)
print("First 10 training labels:", pd.Series(y_train).head(10).tolist())
print("Scaled training pixel range:", np.asarray(x_train).min(), np.asarray(x_train).max())
print("First image, first 10 scaled pixels:", np.asarray(x_train)[0].reshape(-1)[:10])


In [ ]:
# Create model
model = Sequential([
    Dense(128, activation='relu', input_shape=(784,)),
    Dense(10, activation='softmax')
])

# Compile
model.compile(
    optimizer='sgd',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Network architecture:")
model.summary()
print("Optimizer:", type(model.optimizer).__name__)
print("Loss function:", model.loss)


In [ ]:
# Train
history = model.fit(x_train, y_train, epochs=20)
print("Final training epochs:")
print(pd.DataFrame(history.history).tail(3))


In [ ]:
# Evaluate
loss, accuracy = model.evaluate(x_test, y_test)
print("Test loss:", loss)
print("Accuracy:", accuracy * 100, "%")

# Inspect five predictions without retraining
probabilities = model.predict(np.asarray(x_test)[:5], verbose=0)
predicted = probabilities.argmax(axis=1)
actual = np.asarray(y_test)[:5]
print("First test image class probabilities:", probabilities[0].round(3))
print("First five predictions:")
print(pd.DataFrame({"True label": actual, "Predicted label": predicted}))


In [ ]:
# Plot training loss and accuracy
pd.DataFrame(history.history).plot(subplots=True)
